
# MTB AI Predictor — Step 31B (Google Colab + Hugging Face)

This notebook deploys the **locked Step 29 models** through the validated **Step 30 inference engine**.

The Step31A deployment ZIP is downloaded automatically from this Hugging Face **dataset repository**:

- Repository: `jasdeep002/MtbAIM`
- File: `MTB_AI_predictor_v1.zip`
- Repository type: `dataset`
- Revision: `main`

### Outputs
- DS activity classification
- DR activity classification
- DS MIC, MIC90, MIC99 regression
- DR MIC, MIC90, MIC99 regression
- 80%, 90%, and 95% regression residual intervals
- nearest-training-set Morgan/Tanimoto similarity

### Input
- paste one SMILES per line, or
- upload a CSV containing a SMILES column.

**Maximum: 100 molecules per prediction run.**

The Gradio interface is enabled only after:
1. Hugging Face download,
2. bundle SHA256 validation,
3. compact 2265-feature reproduction QC,
4. Step29 smoke-prediction reproduction,
5. 100-SMILES inference stress testing.

> Research use only. This predictor is not a clinical diagnostic.


## 1. Hugging Face deployment source

In [ ]:

#@title Step 31B Hugging Face source
HF_REPO_ID = "jasdeep002/MtbAIM"
HF_FILENAME = "MTB_AI_predictor_v1.zip"
HF_REPO_TYPE = "dataset"
HF_REVISION = "main"

EXPECTED_BUNDLE_NAME = "MTB_AI_predictor_v1.zip"
WORK_DIR = "/content/mtb_ai_predictor"

print("Hugging Face source")
print("  repo_id  :", HF_REPO_ID)
print("  filename :", HF_FILENAME)
print("  repo_type:", HF_REPO_TYPE)
print("  revision :", HF_REVISION)


## 2. Download from Hugging Face, verify, and safely extract

In [ ]:

from pathlib import Path
import shutil, zipfile, stat, subprocess, sys

WORK = Path(WORK_DIR)
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True, exist_ok=True)

EXTRACT_DIR = WORK / "bundle"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Lightweight Hub client only. Model-library versions are installed later from
# the validated Step31A requirements file.
subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--quiet", "--disable-pip-version-check",
        "huggingface_hub>=0.24",
    ],
    check=True,
)

from huggingface_hub import HfApi, hf_hub_download

print("Resolving Hugging Face dataset revision...")
api = HfApi()

try:
    repo_info = api.dataset_info(
        repo_id=HF_REPO_ID,
        revision=HF_REVISION,
    )
except Exception as exc:
    raise RuntimeError(
        f"Cannot access Hugging Face dataset {HF_REPO_ID!r} "
        f"at revision {HF_REVISION!r}: {exc}"
    ) from exc

RESOLVED_HF_COMMIT = getattr(repo_info, "sha", None)
if not RESOLVED_HF_COMMIT:
    raise RuntimeError("Hugging Face did not return a resolved commit SHA.")

available_files = {
    getattr(item, "rfilename", None)
    for item in getattr(repo_info, "siblings", [])
}
if HF_FILENAME not in available_files:
    raise FileNotFoundError(
        f"{HF_FILENAME!r} is not present in Hugging Face dataset "
        f"{HF_REPO_ID!r} at revision {HF_REVISION!r}."
    )

print("Resolved commit:", RESOLVED_HF_COMMIT)
print("Downloading deployment bundle...")

try:
    downloaded_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=HF_FILENAME,
        repo_type=HF_REPO_TYPE,
        revision=RESOLVED_HF_COMMIT,
    )
except Exception as exc:
    raise RuntimeError(
        f"Failed to download {HF_FILENAME!r} from Hugging Face: {exc}"
    ) from exc

ZIP_PATH = Path(downloaded_path)
if not ZIP_PATH.exists() or ZIP_PATH.stat().st_size <= 0:
    raise RuntimeError("Downloaded Hugging Face ZIP is missing or empty.")

print(f"Downloaded: {ZIP_PATH}")
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024**2):.2f} MB")

def _safe_extract_zip(zip_path: Path, destination: Path):
    root = destination.resolve()
    with zipfile.ZipFile(zip_path, "r") as zf:
        bad_crc = zf.testzip()
        if bad_crc is not None:
            raise RuntimeError(f"ZIP CRC validation failed at {bad_crc}")

        infos = zf.infolist()
        if len(infos) > 200:
            raise RuntimeError(
                f"Unexpectedly large bundle file count: {len(infos)}"
            )

        total_uncompressed = sum(int(x.file_size) for x in infos)
        if total_uncompressed > 4 * 1024**3:
            raise RuntimeError("Bundle exceeds 4-GB extraction safety limit.")

        for info in infos:
            mode = (info.external_attr >> 16) & 0o170000
            if mode == stat.S_IFLNK:
                raise RuntimeError(
                    f"Bundle contains symbolic link: {info.filename}"
                )

            target = (destination / info.filename).resolve()
            if target != root and root not in target.parents:
                raise RuntimeError(f"Unsafe ZIP path: {info.filename}")

        zf.extractall(destination)

_safe_extract_zip(ZIP_PATH, EXTRACT_DIR)

config_files = list(EXTRACT_DIR.rglob("deployment_config.json"))
if len(config_files) != 1:
    raise RuntimeError(
        "Expected exactly one deployment_config.json; "
        f"found {len(config_files)}."
    )

BUNDLE_ROOT = config_files[0].parent.resolve()
print("Bundle root:", BUNDLE_ROOT)

validator = BUNDLE_ROOT / "validate_bundle.py"
if not validator.exists():
    raise FileNotFoundError(validator)

# Step31A validator is stdlib-only and validates every packaged SHA256.
proc = subprocess.run(
    [sys.executable, str(validator)],
    cwd=str(BUNDLE_ROOT),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print(proc.stdout)

if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError("Step31A bundle checksum validation failed.")

print("Hugging Face download + ZIP integrity + bundle hashes: PASS")


## 3. Install the exact deployment environment

In [ ]:

import subprocess, sys
from pathlib import Path

requirements = BUNDLE_ROOT / "requirements_colab.txt"
if not requirements.exists():
    raise FileNotFoundError(requirements)

print("Installing deployment dependencies from:")
print(requirements.read_text())

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--quiet",
        "--disable-pip-version-check",
        "-r", str(requirements),
    ],
    check=True,
)

# Confirm the bundle still validates after installation.
post = subprocess.run(
    [sys.executable, str(BUNDLE_ROOT / "validate_bundle.py")],
    cwd=str(BUNDLE_ROOT),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print(post.stdout)
if post.returncode != 0:
    print(post.stderr)
    raise RuntimeError("Bundle validation failed after dependency installation.")

print("Dependency installation: PASS")


## 4. Load the locked Step 30 inference engine and all eight models

In [ ]:

import os, json, importlib.util, sys
from pathlib import Path

# Step30 uses project-relative paths; make the extracted bundle the working root.
os.chdir(BUNDLE_ROOT)

config = json.loads((BUNDLE_ROOT / "deployment_config.json").read_text())
MAX_SMILES = int(config["max_smiles_per_run"])
if MAX_SMILES != 100:
    raise RuntimeError(f"Deployment contract requires max_smiles_per_run=100; found {MAX_SMILES}")

engine_path = BUNDLE_ROOT / config["inference_engine"]
if not engine_path.exists():
    raise FileNotFoundError(engine_path)

spec = importlib.util.spec_from_file_location("mtb_step30_engine", engine_path)
engine = importlib.util.module_from_spec(spec)
sys.modules["mtb_step30_engine"] = engine
spec.loader.exec_module(engine)

bundles, FEATURE_COLUMNS, STEP29_MANIFEST = engine.load_and_validate_step29()

expected_tasks = {
    "DS_CLASSIFICATION",
    "DR_CLASSIFICATION",
    "DS_MIC_REGRESSION",
    "DS_MIC90_REGRESSION",
    "DS_MIC99_REGRESSION",
    "DR_MIC_REGRESSION",
    "DR_MIC90_REGRESSION",
    "DR_MIC99_REGRESSION",
}
if set(bundles) != expected_tasks:
    raise RuntimeError(
        f"Unexpected deployment task set. Missing={sorted(expected_tasks-set(bundles))}, "
        f"extra={sorted(set(bundles)-expected_tasks)}"
    )

if len(FEATURE_COLUMNS) != 2265:
    raise RuntimeError(f"Expected 2265 features, found {len(FEATURE_COLUMNS)}")

print("Model loading: PASS")
print("  Classification: DS + DR")
print("  Regression: DS/DR × MIC/MIC90/MIC99")
print("  Features: 217 RDKit + 2048 Morgan = 2265")
print("  Maximum molecules/run:", MAX_SMILES)


## 5. Deployment QC: reproduce frozen features and Step 29 smoke predictions

In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

QC_RTOL = 1e-6
QC_ATOL = 1e-7

# -------------------------------------------------------------------------
# A. Compact frozen feature reproduction
# -------------------------------------------------------------------------
qc_feature_path = BUNDLE_ROOT / config["compact_feature_qc"]
qc = pd.read_csv(qc_feature_path, compression="infer")

meta_cols = ["dataset", "source_row_index", "row_id", "canonical_smiles"]
if list(qc.columns[:4]) != meta_cols:
    raise RuntimeError("Compact feature-QC metadata columns are not in the expected order.")
if list(map(str, qc.columns[4:])) != list(map(str, FEATURE_COLUMNS)):
    raise RuntimeError("Compact feature-QC feature names/order differ from Step29.")

parsed = engine.parse_feature_contract(FEATURE_COLUMNS)
rd_pos = np.asarray([x[0] for x in parsed[0]], dtype=int)
mg_pos = np.asarray([x[0] for x in parsed[1]], dtype=int)

feature_reports = {}

for dataset in ("DS", "DR"):
    block = qc[qc["dataset"].astype(str) == dataset].reset_index(drop=True)
    if len(block) == 0:
        raise RuntimeError(f"Compact QC contains no {dataset} rows.")

    X_new = np.empty((len(block), 2265), dtype=np.float32)
    for i, smi in enumerate(block["canonical_smiles"].astype(str)):
        mol, canon, status = engine.canonicalize_smiles(smi)
        if mol is None:
            raise RuntimeError(f"Invalid frozen QC SMILES for {dataset}: {smi}")
        X_new[i], _ = engine.molecule_feature_vector(
            mol, FEATURE_COLUMNS, parsed
        )

    raw = block[FEATURE_COLUMNS].apply(
        pd.to_numeric, errors="coerce"
    ).to_numpy(dtype=np.float64)
    X_old, _ = engine.sanitize_numeric_matrix(raw)

    # Morgan bits must be exact.
    if not np.array_equal(
        X_new[:, mg_pos], X_old[:, mg_pos], equal_nan=True
    ):
        raise RuntimeError(f"{dataset} Morgan feature reproduction FAILED.")

    # RDKit descriptors allow only tiny floating point differences.
    if not np.array_equal(np.isnan(X_new[:, rd_pos]), np.isnan(X_old[:, rd_pos])):
        raise RuntimeError(f"{dataset} RDKit NaN pattern reproduction FAILED.")
    if not np.allclose(
        X_new[:, rd_pos],
        X_old[:, rd_pos],
        rtol=QC_RTOL,
        atol=QC_ATOL,
        equal_nan=True,
    ):
        diff = np.abs(
            X_new[:, rd_pos].astype(float) - X_old[:, rd_pos].astype(float)
        )
        diff[np.isnan(diff)] = -1
        r, c = np.unravel_index(np.argmax(diff), diff.shape)
        feature_name = FEATURE_COLUMNS[int(rd_pos[c])]
        raise RuntimeError(
            f"{dataset} RDKit feature reproduction FAILED at {feature_name}: "
            f"new={X_new[r, rd_pos[c]]!r}, frozen={X_old[r, rd_pos[c]]!r}"
        )

    feature_reports[dataset] = len(block)
    print(f"Feature reproduction {dataset}: PASS ({len(block)} rows × 2265 features)")

# -------------------------------------------------------------------------
# B. Step29 smoke-prediction reproduction
# -------------------------------------------------------------------------
smoke_path = BUNDLE_ROOT / config["smoke_prediction_qc"]
smoke = pd.read_csv(smoke_path, compression="infer")

required = {"task", "canonical_smiles", "expected_step29_prediction"}
if not required.issubset(smoke.columns):
    raise RuntimeError(
        f"Smoke reference missing columns: {sorted(required-set(smoke.columns))}"
    )

actual_predictions = []
actual_labels = []

feature_cache = {}

for row in smoke.itertuples(index=False):
    task = str(row.task)
    smi = str(row.canonical_smiles)

    if smi not in feature_cache:
        mol, canon, status = engine.canonicalize_smiles(smi)
        if mol is None:
            raise RuntimeError(f"Invalid smoke-reference SMILES: {smi}")
        vec, _bits = engine.molecule_feature_vector(
            mol, FEATURE_COLUMNS, parsed
        )
        feature_cache[smi] = vec

    vec = feature_cache[smi].reshape(1, -1)
    b = bundles[task]
    Xt = b["imputer"].transform(vec).astype(np.float32, copy=False)

    if task.endswith("_CLASSIFICATION"):
        classes = list(b["model"].classes_)
        pos = b.get("positive_class", 1)
        prob = float(
            b["model"].predict_proba(Xt)[0, classes.index(pos)]
        )
        actual_predictions.append(prob)
        actual_labels.append(
            "Active" if prob >= float(b["threshold"]) else "Non-active"
        )
    else:
        pred = float(np.asarray(b["model"].predict(Xt), dtype=float)[0])
        actual_predictions.append(pred)
        actual_labels.append("")

expected = pd.to_numeric(
    smoke["expected_step29_prediction"], errors="raise"
).to_numpy(dtype=float)
actual = np.asarray(actual_predictions, dtype=float)

if not np.allclose(actual, expected, rtol=1e-6, atol=1e-7):
    absdiff = np.abs(actual - expected)
    i = int(np.argmax(absdiff))
    raise RuntimeError(
        "Step29 smoke-prediction reproduction FAILED: "
        f"task={smoke.iloc[i]['task']}, expected={expected[i]}, "
        f"actual={actual[i]}, absdiff={absdiff[i]}"
    )

if "derived_label" in smoke.columns:
    for i, row in smoke.iterrows():
        if str(row["task"]).endswith("_CLASSIFICATION"):
            expected_label = str(row["derived_label"])
            if expected_label and expected_label.lower() != "nan":
                if actual_labels[i] != expected_label:
                    raise RuntimeError(
                        f"Classification smoke label mismatch for {row['task']}: "
                        f"{actual_labels[i]} != {expected_label}"
                    )

print(
    f"Step29 smoke predictions: PASS "
    f"({len(smoke)} rows across {smoke['task'].nunique()} tasks)"
)
print("DEPLOYMENT QC: PASS")


## 6. Cache applicability-domain references and run a 100-SMILES stress test

In [ ]:

# Applicability-domain fingerprints are calculated once at startup and then
# reused across all Gradio submissions.
TRAINING_FPS = engine.build_training_fps_by_task()

for task in sorted(TRAINING_FPS):
    if len(TRAINING_FPS[task]) == 0:
        raise RuntimeError(f"No training fingerprints were loaded for {task}")

print("Applicability-domain reference cache: PASS")
print("Reference counts:")
for task in sorted(TRAINING_FPS):
    print(f"  {task}: {len(TRAINING_FPS[task]):,}")

# 100-molecule model-path stress test (feature generation + all 8 models).
stress_smiles = (
    ["CCO", "CCN", "c1ccccc1", "CC(=O)O", "CCOC(=O)C"] * 20
)
assert len(stress_smiles) == 100

X_stress, _canon, status_stress, _fps = engine.batch_features(
    stress_smiles, FEATURE_COLUMNS
)
valid_stress = np.asarray(
    [str(s).startswith("OK") for s in status_stress], dtype=bool
)
if valid_stress.sum() != 100:
    raise RuntimeError("100-SMILES stress set unexpectedly contains invalid structures.")

stress_pred = engine.predict_matrix(X_stress, valid_stress, bundles)
if not stress_pred:
    raise RuntimeError("100-SMILES stress test returned no model outputs.")
for name, values in stress_pred.items():
    if len(values) != 100:
        raise RuntimeError(f"Stress output {name} has {len(values)} rows instead of 100.")
    arr = np.asarray(values)
    if arr.dtype.kind in "fc" and not np.isfinite(arr).all():
        raise RuntimeError(f"Stress output {name} contains non-finite values.")

print("100-SMILES inference stress test: PASS")



## 7. Launch the Google Colab / Gradio interface

The deployment bundle has now been downloaded automatically from
`jasdeep002/MtbAIM` on Hugging Face and has passed the deployment QC.

The app supports:

- **Paste mode:** one SMILES per line.
- **CSV mode:** upload a CSV with `SMILES`, `smiles`, `canonical_smiles`, or `standardized_smiles`.
- **Maximum 100 rows per run.**

The on-screen table is compact. The downloadable CSV contains the full pMIC,
nM/µM, 80/90/95% interval, and nearest-training similarity outputs.

No arbitrary applicability-domain cutoff is imposed by default.

When launched in Colab with `share=True`, the Gradio share link remains active
while that Colab runtime is running.


In [ ]:

import gradio as gr
import pandas as pd
import numpy as np
from pathlib import Path
import uuid, time

OUTPUT_DIR = Path("/content/MTB_AI_predictions")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

AD_TASKS = (
    "DS_CLASSIFICATION",
    "DR_CLASSIFICATION",
    "DS_MIC_REGRESSION",
    "DS_MIC90_REGRESSION",
    "DS_MIC99_REGRESSION",
    "DR_MIC_REGRESSION",
    "DR_MIC90_REGRESSION",
    "DR_MIC99_REGRESSION",
)

def _uploaded_path(obj):
    if obj is None:
        return None
    if isinstance(obj, str):
        return obj
    if hasattr(obj, "name"):
        return obj.name
    return str(obj)

def _prepare_input(pasted_smiles, csv_file):
    text = (pasted_smiles or "").strip()
    file_path = _uploaded_path(csv_file)

    if text and file_path:
        raise ValueError("Use either pasted SMILES or a CSV upload, not both.")

    if file_path:
        df = pd.read_csv(file_path)
        if len(df) == 0:
            raise ValueError("Uploaded CSV contains zero rows.")
        smiles_col = engine.infer_smiles_column(df, None)
    elif text:
        smiles = [line.strip() for line in text.splitlines() if line.strip()]
        if not smiles:
            raise ValueError("No SMILES were supplied.")
        df = pd.DataFrame({
            "Compound_ID": [f"Compound_{i+1:03d}" for i in range(len(smiles))],
            "SMILES": smiles,
        })
        smiles_col = "SMILES"
    else:
        raise ValueError("Paste SMILES or upload a CSV.")

    if len(df) > MAX_SMILES:
        raise ValueError(
            f"Maximum {MAX_SMILES} molecules per run. "
            f"This submission contains {len(df)} rows."
        )

    return df.reset_index(drop=True), smiles_col

def _predict_for_ui(pasted_smiles, csv_file):
    try:
        df, smiles_col = _prepare_input(pasted_smiles, csv_file)

        X, canonical, status, query_fps = engine.batch_features(
            df[smiles_col].tolist(), FEATURE_COLUMNS
        )
        valid = np.asarray(
            [str(s).startswith("OK") for s in status], dtype=bool
        )

        additions = {
            "step30_canonical_smiles": canonical,
            "step30_smiles_status": status,
        }
        additions.update(engine.predict_matrix(X, valid, bundles))

        # Reuse the precomputed task-specific training fingerprints.
        for task in AD_TASKS:
            sims = np.full(len(df), np.nan, dtype=float)
            for i, bits in enumerate(query_fps):
                if bits is None:
                    continue
                sims[i] = engine.nearest_training_similarity(
                    bits, TRAINING_FPS[task]
                )
            additions[f"{task}_max_train_tanimoto"] = sims

        full = pd.concat(
            [df, pd.DataFrame(additions)],
            axis=1,
            copy=False,
        )

        stamp = time.strftime("%Y%m%d_%H%M%S")
        token = uuid.uuid4().hex[:8]
        out_path = OUTPUT_DIR / f"MTB_AI_predictions_{stamp}_{token}.csv"
        full.to_csv(out_path, index=False)

        summary = pd.DataFrame({
            "Row": np.arange(1, len(full) + 1),
            "Input SMILES": df[smiles_col].astype(str),
            "Status": full["step30_smiles_status"],
            "DS activity": full.get("DS_activity_call"),
            "DS P(active)": full.get("DS_probability_active"),
            "DR activity": full.get("DR_activity_call"),
            "DR P(active)": full.get("DR_probability_active"),
            "DS MIC (uM)": full.get("DS_MIC_uM"),
            "DS MIC90 (uM)": full.get("DS_MIC90_uM"),
            "DS MIC99 (uM)": full.get("DS_MIC99_uM"),
            "DR MIC (uM)": full.get("DR_MIC_uM"),
            "DR MIC90 (uM)": full.get("DR_MIC90_uM"),
            "DR MIC99 (uM)": full.get("DR_MIC99_uM"),
            "DS train similarity": full.get(
                "DS_CLASSIFICATION_max_train_tanimoto"
            ),
            "DR train similarity": full.get(
                "DR_CLASSIFICATION_max_train_tanimoto"
            ),
        })

        # Display-only formatting. Full-precision values remain in the CSV.
        for col in ("DS P(active)", "DR P(active)",
                    "DS train similarity", "DR train similarity"):
            summary[col] = pd.to_numeric(
                summary[col], errors="coerce"
            ).round(4)

        for col in [
            "DS MIC (uM)", "DS MIC90 (uM)", "DS MIC99 (uM)",
            "DR MIC (uM)", "DR MIC90 (uM)", "DR MIC99 (uM)",
        ]:
            summary[col] = pd.to_numeric(
                summary[col], errors="coerce"
            ).map(lambda x: np.nan if pd.isna(x) else float(f"{x:.4g}"))

        n_valid = int(valid.sum())
        n_invalid = int(len(df) - n_valid)
        n_multi = int(
            (full["step30_smiles_status"] == "OK_MULTI_FRAGMENT").sum()
        )

        message = (
            f"✅ **Prediction complete.** "
            f"Rows: **{len(df)}** | Valid: **{n_valid}** | "
            f"Invalid: **{n_invalid}** | Multi-fragment: **{n_multi}**.  \n"
            "The downloadable CSV contains the complete pMIC, nM/µM, "
            "80/90/95% interval, and applicability-similarity output.  \n"
            "**Research use only; not a clinical diagnostic.**"
        )
        return summary, str(out_path), message

    except Exception as exc:
        return (
            pd.DataFrame(),
            None,
            f"❌ **Prediction failed:** `{type(exc).__name__}: {exc}`"
        )

with gr.Blocks(title="MTB AI Predictor") as demo:
    gr.Markdown(
        """
# MTB AI Predictor
Predict activity and quantitative MIC-family endpoints for *Mycobacterium tuberculosis*.

**Models:** DS and DR classification + separate DS/DR MIC, MIC90, and MIC99 regressors.  
**Limit:** up to **100 molecules per run**.  
**Input:** paste one SMILES per line **or** upload a CSV.  
**Research use only — not a clinical diagnostic.**
        """
    )

    with gr.Row():
        with gr.Column():
            pasted = gr.Textbox(
                label="Paste SMILES (one per line)",
                lines=12,
                placeholder="CCO\nCCN\nc1ccccc1",
            )
        with gr.Column():
            csv_input = gr.File(
                label="Or upload CSV",
                file_types=[".csv"],
                type="filepath",
            )
            gr.Markdown(
                "Recognized SMILES columns: `SMILES`, `smiles`, "
                "`canonical_smiles`, `standardized_smiles`."
            )

    predict_btn = gr.Button("Predict", variant="primary")
    clear_btn = gr.ClearButton([pasted, csv_input])

    status_box = gr.Markdown()
    table = gr.Dataframe(
        label="Prediction summary",
        interactive=False,
        wrap=True,
    )
    download = gr.File(label="Download full prediction CSV")

    predict_btn.click(
        fn=_predict_for_ui,
        inputs=[pasted, csv_input],
        outputs=[table, download, status_box],
    )

demo.queue()
demo.launch(
    share=True,
    inline=True,
    show_error=True,
)
